# Download Tokamark Sample Data from S3

This notebook downloads Zarr files from a remote S3-compatible storage using shots from the tokamark data splits.

## Overview
- Uses tokamark library to get train/test/val shot splits
- Connects to S3 storage (STFC Echo)
- Downloads selected shots locally with error handling

## Requirements
```bash
pip install fsspec s3fs tokamark
```

## Configuration

In [1]:
import os
from pathlib import Path
from typing import List

# Configuration
S3_ENDPOINT = "https://s3.echo.stfc.ac.uk"
S3_BUCKET_PATH = "mast/tokamark/v1"
LOCAL_DATA_DIR = "./data"

# Create local directory if it doesn't exist
Path(LOCAL_DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"✓ Data directory: {Path(LOCAL_DATA_DIR).resolve()}")

✓ Data directory: /rds/project/rds-mOlK9qn0PlQ/ir-rous1/output/cnn-baseline/data


## Connect to S3 Storage

In [2]:
import fsspec

try:
    fs = fsspec.filesystem(
        "s3",
        client_kwargs={"endpoint_url": S3_ENDPOINT},
        anon=True  # Public data access
    )
    print("✓ Connected to S3 storage successfully")
except Exception as e:
    print(f"✗ Failed to connect to S3: {e}")
    raise

✓ Connected to S3 storage successfully


## Get Shots from Tokamark Data Splits

In [6]:
from tokamark.tools.path import RANDOM_SPLIT_TOKAMARK_DATA_SPLITS_FILE
from tokamark.data_split import get_train_test_val_shots

try:
    train_shots_, test_shots_, val_shots_ = get_train_test_val_shots(
        max_index=8,
        shuffle=True,
        data_splits_file_path=RANDOM_SPLIT_TOKAMARK_DATA_SPLITS_FILE        
    )
    
    print("✓ Retrieved shots from tokamark data splits\n")
    print(f"  Train shots: {len(train_shots_)} - {train_shots_}")
    print(f"  Test shots:  {len(test_shots_)} - {test_shots_}")
    print(f"  Val shots:   {len(val_shots_)} - {val_shots_}")
    
except Exception as e:
    print(f"✗ Failed to get shots from tokamark: {e}")
    raise

✓ Retrieved shots from tokamark data splits

  Train shots: 8 - [27526, 25659, 24885, 23302, 15245, 30048, 22181, 15971]
  Test shots:  8 - [12413, 20284, 19387, 26954, 20333, 16422, 17080, 11849]
  Val shots:   8 - [27009, 21769, 20127, 15656, 14144, 29411, 15490, 21725]


## Select Which Splits to Download

In [7]:
# Choose which splits to download
# Set to True to include that split
include_train = True
include_test = True
include_val = True

# Combine selected shots
selected_shots = []
split_names = []

if include_train:
    selected_shots.extend(train_shots_)
    split_names.append(f"train ({len(train_shots_)})")

if include_test:
    selected_shots.extend(test_shots_)
    split_names.append(f"test ({len(test_shots_)})")

if include_val:
    selected_shots.extend(val_shots_)
    split_names.append(f"val ({len(val_shots_)})")

selected_shots = list(set(selected_shots))  # Remove duplicates
selected_shots.sort()

print(f"✓ Selected splits: {', '.join(split_names)}")
print(f"✓ Total unique shots: {len(selected_shots)}\n")
print("Shots to download:")
for i, shot in enumerate(selected_shots, 1):
    print(f"  {i:2d}. Shot {shot}")

✓ Selected splits: train (8), test (8), val (8)
✓ Total unique shots: 24

Shots to download:
   1. Shot 11849
   2. Shot 12413
   3. Shot 14144
   4. Shot 15245
   5. Shot 15490
   6. Shot 15656
   7. Shot 15971
   8. Shot 16422
   9. Shot 17080
  10. Shot 19387
  11. Shot 20127
  12. Shot 20284
  13. Shot 20333
  14. Shot 21725
  15. Shot 21769
  16. Shot 22181
  17. Shot 23302
  18. Shot 24885
  19. Shot 25659
  20. Shot 26954
  21. Shot 27009
  22. Shot 27526
  23. Shot 29411
  24. Shot 30048


## Find Shots in S3 Storage

In [8]:
try:
    files = fs.ls(S3_BUCKET_PATH)
    zarr_files = sorted([f for f in files if f.endswith(".zarr")])
    
    print(f"✓ Found {len(zarr_files)} Zarr files in S3\n")
    
    # Map shot numbers to file paths
    def get_shot_number(zarr_path: str) -> int:
        return int(zarr_path.split("/")[-1].replace(".zarr", ""))
    
    shot_to_path = {get_shot_number(f): f for f in zarr_files}
    
    # Find which selected shots exist in S3
    found_shots = []
    missing_shots = []
    
    for shot in selected_shots:
        if shot in shot_to_path:
            found_shots.append(shot)
        else:
            missing_shots.append(shot)
    
    print(f"✓ Found {len(found_shots)} shots in S3")
    if missing_shots:
        print(f"✗ Missing {len(missing_shots)} shots in S3: {missing_shots}")
    
    # Create list of files to download
    selected_files = [shot_to_path[shot] for shot in found_shots]
        
except Exception as e:
    print(f"✗ Failed to find shots in S3: {e}")
    raise

✓ Found 11573 Zarr files in S3

✓ Found 24 shots in S3


## Download Files with Progress Tracking

In [9]:
def download_file(fs, remote_path: str, local_dir: str) -> tuple[bool, str]:
    """
    Download a single file from remote storage.
    
    Args:
        fs: fsspec filesystem object
        remote_path: Full remote path to the file
        local_dir: Local directory to save to
    
    Returns:
        Tuple of (success: bool, message: str)
    """
    try:
        shot_name = remote_path.split("/")[-1]
        local_path = os.path.join(local_dir, shot_name)
        
        # Skip if already exists
        if os.path.exists(local_path):
            return True, f"Already exists: {shot_name}"
        
        fs.get(remote_path, local_path, recursive=True)
        return True, f"Downloaded: {shot_name}"
        
    except Exception as e:
        return False, f"Error: {str(e)}"


# Download files with summary
results = {"success": 0, "skipped": 0, "failed": 0, "errors": []}

print(f"Downloading {len(selected_files)} files...\n")

for i, remote_path in enumerate(selected_files, 1):
    shot_name = remote_path.split("/")[-1]
    success, message = download_file(fs, remote_path, LOCAL_DATA_DIR)
    
    status = "✓" if success else "✗"
    print(f"[{i}/{len(selected_files)}] {status} {message}")
    
    if success:
        if "Already" in message:
            results["skipped"] += 1
        else:
            results["success"] += 1
    else:
        results["failed"] += 1
        results["errors"].append(message)


[1/24] ✓ Downloaded: 11849.zarr
[2/24] ✓ Downloaded: 12413.zarr
[3/24] ✓ Downloaded: 14144.zarr
[4/24] ✓ Downloaded: 15245.zarr
[5/24] ✓ Downloaded: 15490.zarr
[6/24] ✓ Downloaded: 15656.zarr
[7/24] ✓ Downloaded: 15971.zarr
[8/24] ✓ Downloaded: 16422.zarr
[9/24] ✓ Downloaded: 17080.zarr
[10/24] ✓ Downloaded: 19387.zarr
[11/24] ✓ Downloaded: 20127.zarr
[12/24] ✓ Downloaded: 20284.zarr
[13/24] ✓ Downloaded: 20333.zarr
[14/24] ✓ Downloaded: 21725.zarr
[15/24] ✓ Downloaded: 21769.zarr
[16/24] ✓ Downloaded: 22181.zarr
[17/24] ✓ Downloaded: 23302.zarr
[18/24] ✓ Downloaded: 24885.zarr
[19/24] ✓ Downloaded: 25659.zarr
[20/24] ✓ Downloaded: 26954.zarr
[21/24] ✓ Downloaded: 27009.zarr
[22/24] ✓ Downloaded: 27526.zarr
[23/24] ✓ Downloaded: 29411.zarr
[24/24] ✓ Downloaded: 30048.zarr


## Download Summary

In [ ]:
print("\n" + "="*50)
print("DOWNLOAD SUMMARY")
print("="*50)
print(f"✓ Successful:  {results['success']}")
print(f"⊘ Skipped:    {results['skipped']}")
print(f"✗ Failed:     {results['failed']}")
print("="*50)

if results['errors']:
    print("\nErrors encountered:")
    for error in results['errors']:
        print(f"  • {error}")

# List downloaded files
downloaded_files = list(Path(LOCAL_DATA_DIR).glob("*.zarr"))
print(f"\nLocal files in {LOCAL_DATA_DIR}: {len(downloaded_files)}")
if downloaded_files:
    print("\nFiles:")
    for f in sorted(downloaded_files):
        size_mb = f.stat().st_size / (1024**2)
        print(f"  • {f.name:20s} ({size_mb:3f} MB)")


DOWNLOAD SUMMARY
✓ Successful:  24
⊘ Skipped:    0
✗ Failed:     0

Local files in ./data: 34

Files:
  • 11849.zarr           (0.003906 MB)
  • 12413.zarr           (0.003906 MB)
  • 13686.zarr           (0.003906 MB)
  • 14144.zarr           (0.003906 MB)
  • 15245.zarr           (0.003906 MB)
  • 15490.zarr           (0.003906 MB)
  • 15656.zarr           (0.003906 MB)
  • 15971.zarr           (0.003906 MB)
  • 16422.zarr           (0.003906 MB)
  • 17080.zarr           (0.003906 MB)
  • 17393.zarr           (0.003906 MB)
  • 19387.zarr           (0.003906 MB)
  • 20127.zarr           (0.003906 MB)
  • 20284.zarr           (0.003906 MB)
  • 20333.zarr           (0.003906 MB)
  • 21725.zarr           (0.003906 MB)
  • 21752.zarr           (0.003906 MB)
  • 21769.zarr           (0.003906 MB)
  • 22181.zarr           (0.003906 MB)
  • 22381.zarr           (0.003906 MB)
  • 22679.zarr           (0.003906 MB)
  • 22709.zarr           (0.003906 MB)
  • 23302.zarr           (0.003906 MB)
